In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

# Grid parameters
nx = 100
nt = 50
L = 1.0
T = 1.0
dx = L / (nx-1)
dt = T / nt
x = np.linspace(0, L, nx)

# Physical parameters
c = 1.0
D = 0.01

# Initial condition
u0 = np.exp(-200*(x-0.5)**2)

# Centered advection
def advect_centered(u, dt, dx, c):
    un = u.copy()
    un[1:-1] = u[1:-1] - c*dt/(2*dx)*(u[2:] - u[:-2])
    return un

# Crank-Nicolson diffusion
def diffuse_CN(u, dt, dx, D):
    nx = len(u)
    r = D*dt/(2*dx**2)
    A = np.diag((1+2*r)*np.ones(nx)) + np.diag(-r*np.ones(nx-1),1) + np.diag(-r*np.ones(nx-1),-1)
    B = np.diag((1-2*r)*np.ones(nx)) + np.diag(r*np.ones(nx-1),1) + np.diag(r*np.ones(nx-1),-1)
    return np.linalg.solve(A, B @ u)

# Forward simulations
u_adv_only = [u0.copy()]
u_diff_only = [u0.copy()]
u_adv_diff = [u0.copy()]

u_a = u0.copy()
u_d = u0.copy()
u_ad = u0.copy()

for n in range(nt):
    # Forward advection only
    u_a = advect_centered(u_a, dt, dx, c)
    u_adv_only.append(u_a.copy())
    
    # Forward diffusion only
    u_d = diffuse_CN(u_d, dt, dx, D)
    u_diff_only.append(u_d.copy())
    
    # Forward advection + diffusion
    u_ad = advect_centered(u_ad, dt, dx, c)
    u_ad = diffuse_CN(u_ad, dt, dx, D)
    u_adv_diff.append(u_ad.copy())

# Reverse simulations
u_rev_adv = [u_adv_diff[-1].copy()]
u_rev_diff = [u_adv_diff[-1].copy()]
u_rev_total = [u_adv_diff[-1].copy()]

u_ra = u_adv_diff[-1].copy()
u_rd = u_adv_diff[-1].copy()
u_rt = u_adv_diff[-1].copy()

for n in range(nt):
    u_ra = advect_centered(u_ra, -dt, dx, c)
    u_rev_adv.append(u_ra.copy())
    
    u_rd = diffuse_CN(u_rd, -dt, dx, D)
    u_rev_diff.append(np.clip(u_rd, -0.5, 1.5))
    
    u_rt = advect_centered(u_rt, -dt, dx, c)
    u_rt = diffuse_CN(u_rt, -dt, dx, D)
    u_rev_total.append(np.clip(u_rt, -0.5, 1.5))

# Function to compute dynamic y-limits with margin
def get_ylim(data_list, margin=0.05):
    all_data = np.array(data_list)
    dmin = np.min(all_data)
    dmax = np.max(all_data)
    yrange = dmax - dmin
    return dmin - margin*yrange, dmax + margin*yrange

# Set up figure 2x3
fig, axs = plt.subplots(2,3, figsize=(15,6))

# Upper row: forward
titles_fwd = ['Forward Advection', 'Forward Diffusion', 'Forward Advection+Diffusion']
data_fwd = [u_adv_only, u_diff_only, u_adv_diff]
lines_fwd = []

for i in range(3):
    ylim = get_ylim(data_fwd[i])
    axs[0,i].set_xlim(0,L)
    axs[0,i].set_ylim(ylim)
    axs[0,i].set_xlabel('x')
    axs[0,i].set_ylabel('u(x)')
    axs[0,i].set_title(titles_fwd[i])
    line, = axs[0,i].plot([], [], lw=2)
    lines_fwd.append(line)

# Lower row: reverse
titles_rev = ['Reverse Advection', 'Reverse Diffusion', 'Reverse Adv+Diffusion']
data_rev = [u_rev_adv, u_rev_diff, u_rev_total]
lines_rev = []

for i in range(3):
    ylim = get_ylim(data_rev[i])
    axs[1,i].set_xlim(0,L)
    axs[1,i].set_ylim(ylim)
    axs[1,i].set_xlabel('x')
    axs[1,i].set_ylabel('u(x)')
    axs[1,i].set_title(titles_rev[i])
    line, = axs[1,i].plot([], [], lw=2)
    lines_rev.append(line)

fig.subplots_adjust(hspace=0.5, wspace=0.3)

def init():
    for line in lines_fwd + lines_rev:
        line.set_data([], [])
    return lines_fwd + lines_rev

def update(frame):
    for i in range(3):
        lines_fwd[i].set_data(x, data_fwd[i][frame])
        lines_rev[i].set_data(x, data_rev[i][frame])
    return lines_fwd + lines_rev

anim = FuncAnimation(fig, update, frames=nt+1, init_func=init, blit=True, interval=200)

# Save MP4
writer = FFMpegWriter(fps=5, metadata=dict(artist='ChatGPT'), bitrate=1800)
anim.save('advection_diffusion_reversibility.mp4', writer=writer)

plt.close(fig)
print("Animation saved as 'advection_diffusion_3x2_corrected_ylim.mp4'")


Animation saved as 'advection_diffusion_3x2_corrected_ylim.mp4'
